In [76]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

In [77]:
df=pd.read_csv('Loan_Default.csv')
df.head()

,ID,year,loan_limit,Gender,approv_in_adv,loan_type,loan_purpose,Credit_Worthiness,open_credit,business_or_commercial,...,credit_type,Credit_Score,co-applicant_credit_type,age,submission_of_application,LTV,Region,Security_Type,Status,dtir1
0,24890,2019,cf,Sex Not Available,nopre,type1,p1,l1,nopc,nob/c,...,EXP,758,CIB,25-34,to_inst,98.728814,south,direct,1,45.0
1,24891,2019,cf,Male,nopre,type2,p1,l1,nopc,b/c,...,EQUI,552,EXP,55-64,to_inst,NaN,North,direct,1,NaN
2,24892,2019,cf,Male,pre,type1,p1,l1,nopc,nob/c,...,EXP,834,CIB,35-44,to_inst,80.019685,south,direct,0,46.0
3,24893,2019,cf,Male,nopre,type1,p4,l1,nopc,nob/c,...,EXP,587,CIB,45-54,not_inst,69.376900,North,direct,0,42.0
4,24894,2019,cf,Joint,pre,type1,p1,l1,nopc,nob/c,...,CRIF,602,EXP,25-34,not_inst,91.886544,North,direct,0,39.0


In [78]:
selected_features = ['Credit_Score', 'LTV', 'dtir1', 'loan_type', 'age', 'Region']
X = df[selected_features]
y = df['Status']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42
)

In [79]:

numeric_features = ['Credit_Score', 'LTV', 'dtir1']
categorical_features = ['loan_type', 'age', 'Region']

numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore'))
])

preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, numeric_features),
        ('cat', categorical_transformer, categorical_features)
    ]
)

print("Preprocessor Pipeline built successfully!")

Preprocessor Pipeline built successfully!


In [80]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier

pipelines = {
    "Logistic_Regression": Pipeline(steps=[
        ('preprocessor', preprocessor),
        ('classifier', LogisticRegression(class_weight='balanced', max_iter=1000, random_state=42))
    ]),
    
    "Random_Forest": Pipeline(steps=[
        ('preprocessor', preprocessor),
        ('classifier', RandomForestClassifier(class_weight='balanced', random_state=42))
    ]),
    
    "XGBoost": Pipeline(steps=[
        ('preprocessor', preprocessor),
        ('classifier', XGBClassifier(random_state=42, eval_metric='logloss'))
    ])
}

param_grids = {
    "Logistic_Regression": {
        'classifier__C': [0.1, 1.0, 10.0],
        'classifier__solver': ['liblinear', 'lbfgs']
    },
    "Random_Forest": {
        'classifier__n_estimators': [50, 100, 200],
        'classifier__max_depth': [None, 10, 20]
    },
   "XGBoost": {
        'classifier__n_estimators': [100, 150, 232], 
        'classifier__learning_rate': [0.01, 0.1, 0.0223],
        'classifier__max_depth': [3, 6, 9]
    }
}

print("Model pipelines and hyperparameter working perfectly")

Model pipelines and hyperparameter working perfectly


In [81]:
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import pandas as pd

In [82]:
results = []
best_model_overall = None
best_accuracy_overall = 0
best_model_name = ""
best_params_overall = {}

In [83]:
for model_name, pipeline in pipelines.items():
    print(f"Training {model_name}...")
    
    grid_search = GridSearchCV(
        estimator=pipeline,
        param_grid=param_grids[model_name],
        cv=3,
        scoring='f1',
        n_jobs=-1
    )
    
    grid_search.fit(X_train, y_train)
    
    best_model = grid_search.best_estimator_
    acc = best_model.score(X_test, y_test)
    
    results.append({
        "Model": model_name,
        "Best Parameters": grid_search.best_params_,
        "Test Accuracy": acc
    })
    
    if acc > best_accuracy_overall:
        best_accuracy_overall = acc
        best_model_overall = best_model
        best_model_name = model_name
        best_params_overall = grid_search.best_params_

Training Logistic_Regression...
Training Random_Forest...
Training XGBoost...


In [84]:
performance_df = pd.DataFrame(results)

print("MODEL PERFORMANCE TABLE")
print(performance_df.to_string(index=False))

MODEL PERFORMANCE TABLE
              Model                                                                                 Best Parameters  Test Accuracy
Logistic_Regression                                          {'classifier__C': 10.0, 'classifier__solver': 'lbfgs'}       0.591646
      Random_Forest                                  {'classifier__max_depth': 10, 'classifier__n_estimators': 100}       0.849734
            XGBoost {'classifier__learning_rate': 0.1, 'classifier__max_depth': 6, 'classifier__n_estimators': 150}       0.872839


In [85]:
from sklearn.metrics import classification_report, confusion_matrix

print(f"WINNING MODEL: {best_model_name}")

y_pred_best = best_model_overall.predict(X_test)

print(f"1. Best Hyperparameters:{best_params_overall}")
print(f"2. Accuracy Score:{best_accuracy_overall:.4f}")

print("3. Classification Report:")
print(classification_report(y_test, y_pred_best))

print("4. Confusion Matrix:")
print(confusion_matrix(y_test, y_pred_best))

WINNING MODEL: XGBoost
1. Best Hyperparameters:{'classifier__learning_rate': 0.1, 'classifier__max_depth': 6, 'classifier__n_estimators': 150}
2. Accuracy Score:0.8728
3. Classification Report:
              precision    recall  f1-score   support

           0       0.86      0.99      0.92     22494
           1       0.94      0.51      0.66      7240

    accuracy                           0.87     29734
   macro avg       0.90      0.75      0.79     29734
weighted avg       0.88      0.87      0.86     29734

4. Confusion Matrix:
[[22236   258]
 [ 3523  3717]]


In [86]:
import joblib

model_filename = "best_model.pkl"

joblib.dump(best_model_overall, model_filename)

print(f"Chunk 7 Complete: Model successfully saved as {model_filename}!")

Chunk 7 Complete: Model successfully saved as best_model.pkl!


In [87]:
import optuna
from sklearn.model_selection import cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
import warnings

In [88]:
def objective(trial):
    classifier_name = trial.suggest_categorical("classifier", ["LogisticRegression", "RandomForest", "XGBoost"])
    
    if classifier_name == "LogisticRegression":
        C = trial.suggest_float("C", 0.01, 100.0, log=True)
        solver = trial.suggest_categorical("solver", ["liblinear", "lbfgs"])
        classifier = LogisticRegression(C=C, solver=solver, class_weight='balanced', max_iter=1000, random_state=42)
        
    elif classifier_name == "RandomForest":
        n_estimators = trial.suggest_int("n_estimators", 50, 300)
        max_depth = trial.suggest_int("max_depth", 5, 30)
        min_samples_split = trial.suggest_int("min_samples_split", 2, 10)
        classifier = RandomForestClassifier(
            n_estimators=n_estimators,
            max_depth=max_depth,
            min_samples_split=min_samples_split,
            class_weight='balanced',
            random_state=42
        )
        
    else:
        n_estimators = trial.suggest_int("xgb_n_estimators", 50, 300)
        learning_rate = trial.suggest_float("learning_rate", 0.01, 0.3, log=True)
        max_depth = trial.suggest_int("xgb_max_depth", 3, 10)
        classifier = XGBClassifier(
            n_estimators=n_estimators,
            learning_rate=learning_rate,
            max_depth=max_depth,
            random_state=42,
            eval_metric='logloss'
        )

    pipeline = Pipeline(steps=[
        ('preprocessor', preprocessor),
        ('classifier', classifier)
    ])
    
    score = cross_val_score(pipeline, X_train, y_train, cv=3, scoring='accuracy', n_jobs=-1)
    
    return score.mean()

In [89]:
study = optuna.create_study(direction="maximize", study_name="Finding the Best Classifier and Hyperparameters using Optuna")

print("Starting Optuna optimization (30 trials)...")
study.optimize(objective, n_trials=30)

print("OPTUNA OPTIMIZATION FINISHED!")
print(f"Best Algorithm and Parameters:\n{study.best_params}")
print(f"Best Cross-Validation Accuracy:\n{study.best_value:.4f}")

[I 2026-03-01 12:54:42,501] A new study created in memory with name: Finding the Best Classifier and Hyperparameters using Optuna


Starting Optuna optimization (30 trials)...


[I 2026-03-01 12:55:07,890] Trial 0 finished with value: 0.8588148302030288 and parameters: {'classifier': 'RandomForest', 'n_estimators': 268, 'max_depth': 21, 'min_samples_split': 9}. Best is trial 0 with value: 0.8588148302030288.
[I 2026-03-01 12:55:22,426] Trial 1 finished with value: 0.8554600875502433 and parameters: {'classifier': 'RandomForest', 'n_estimators': 231, 'max_depth': 12, 'min_samples_split': 9}. Best is trial 0 with value: 0.8588148302030288.
[I 2026-03-01 12:55:23,846] Trial 2 finished with value: 0.8708717334859889 and parameters: {'classifier': 'XGBoost', 'xgb_n_estimators': 132, 'learning_rate': 0.17559785443896153, 'xgb_max_depth': 7}. Best is trial 2 with value: 0.8708717334859889.
[I 2026-03-01 12:55:32,481] Trial 3 finished with value: 0.8615137639017991 and parameters: {'classifier': 'RandomForest', 'n_estimators': 81, 'max_depth': 28, 'min_samples_split': 2}. Best is trial 2 with value: 0.8708717334859889.
[I 2026-03-01 12:55:35,669] Trial 4 finished with

OPTUNA OPTIMIZATION FINISHED!
Best Algorithm and Parameters:
{'classifier': 'XGBoost', 'xgb_n_estimators': 53, 'learning_rate': 0.2814996314998899, 'xgb_max_depth': 5}
Best Cross-Validation Accuracy:
0.8721
